### Previewing the Training Data

Let's load and preview the `train.json` dataset to understand its structure and contents.

## ⌛️ Time is ticking. Good luck!

In [1]:
"""
===========================================================
FACTUALITY CLASSIFICATION MODEL — OPTIMIZED FAST LOGISTIC (Balanced)
===========================================================

Goal:
----
Train a text classification model to predict whether an
OpenAI/chatbot answer is:
  • factual
  • contradictory
  • irrelevant

Approach:
---------
We use a competition-optimized, fast pipeline tuned for a
custom weighted confusion-matrix ranking where each class
is equally important (33.3% each). This version uses
Logistic Regression (balanced) for speed and stable probabilities.
"""

# =============================
# 📦 1. Import Dependencies
# =============================
import pandas as pd
import json
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# ==========================================================
# 📥 2. Load Data
# ==========================================================
train_path = "data/train.json"
test_path  = "data/test.json"

with open(train_path, 'r', encoding='utf-8') as f:
    train_df = pd.DataFrame(json.load(f))

with open(test_path, 'r', encoding='utf-8') as f:
    test_df = pd.DataFrame(json.load(f))

print(f"✅ Train samples: {len(train_df)} | Test samples: {len(test_df)}")

# ==========================================================
# 🧹 3. Clean and Prepare the Training Data (PERFORMING EDA)
# ==========================================================
train_df.head(10)
train_df.isna().value_counts()
train_df.info()

train_df = train_df.dropna(subset=['type'])
train_df['type'] = train_df['type'].astype(str).str.strip()

# ==========================================================
# 🧩 4. Combine Text Columns into a Single Feature
# ==========================================================
def combine_text(df):
    return (
        df['question'].fillna('') + ' [Q] ' +
        df['context'].fillna('') + ' [C] ' +
        df['answer'].fillna('')
    )

X_train = combine_text(train_df)
X_test  = combine_text(test_df)

# ==========================================================
# 🏷️ 5. Encode Labels
# ==========================================================
le = LabelEncoder()
y_train = le.fit_transform(train_df['type'])
print(f"Encoded labels: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ==========================================================
# 🧠 6. Define the Fast, Optimized Pipeline (Logistic)
# ==========================================================
RANDOM_STATE = 42

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=4000,
        ngram_range=(1,2),
        lowercase=True,
        stop_words='english',
        sublinear_tf=True
    )),
    ('sel', SelectKBest(chi2, k=2500)),
    ('clf', LogisticRegression(
        solver='saga',
        penalty='l2',
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# ==========================================================
# 🎯 7. Hyperparameter Tuning (Small Grid, Fast CV)
# ==========================================================
param_grid = {
    'clf__C': [0.5, 1.0],
    'tfidf__max_df': [0.9, 0.95]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("\n🔍 Training optimized fast model (keeps runtime low)...")
gs.fit(X_train, y_train)

best_model = gs.best_estimator_
print("\n✅ Best Parameters Found:", gs.best_params_)
print("🚀 Optimized training complete!")

# ==========================================================
# 🤖 8. Make Predictions on Test Set
# ==========================================================
y_pred = best_model.predict(X_test)
test_df['predicted_type'] = le.inverse_transform(y_pred)

print("\n🔮 Sample Predictions:")
print(test_df[['question', 'predicted_type']].head())

# ==========================================================
# 📊 9. Optional: Evaluation Check (No per-class prints)
# ==========================================================
if 'type' in test_df.columns:
    test_df['type'] = test_df['type'].astype(str)
    mask = test_df['type'].isin(le.classes_)

    if mask.any():
        print("\nℹ️ Labels found in test set (evaluation skipped by request).")
    else:
        print("\n⚠️ No matching labels in test set.")
else:
    print("\nℹ️ No 'type' column found in test set — skipping evaluation.")


# ==========================================================
# 💾 10. SAVE FULL TEST DATAFRAME + PREDICTIONS (OPTION A2)
# ==========================================================
# Save the entire test dataframe (original columns + predicted_type)
output_path = "predictions/test_with_predictions.json"

# ensure the folder exists
import os
os.makedirs("predictions", exist_ok=True)

test_df.to_json(output_path, orient="records", indent=2, force_ascii=False)

print(f"\n💾 Saved extended predictions file to:")
print(f"   → {output_path}")

# ==========================================================
# 🏁 End of Script
# ==========================================================


✅ Train samples: 21021 | Test samples: 2000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21021 entries, 0 to 21020
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   answer    21021 non-null  object
 1   type      21021 non-null  object
 2   context   21021 non-null  object
 3   question  21021 non-null  object
dtypes: object(4)
memory usage: 657.0+ KB
Encoded labels: {'contradiction': np.int64(0), 'factual': np.int64(1), 'irrelevant': np.int64(2)}

🔍 Training optimized fast model (keeps runtime low)...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

✅ Best Parameters Found: {'clf__C': 0.5, 'tfidf__max_df': 0.9}
🚀 Optimized training complete!

🔮 Sample Predictions:
                                            question predicted_type
0  What was the Bronx called in the mid-19th cent...        factual
1   When did Beyoncé begin to manage the girl group?        factual
2            To what Roman god was Dionysu